# 04 — Evaluation: Backtest, Scenarios, Dashboard

Backtest on the test period only; 5 bps cost per position change. Scenario engine: deterministic bull/base/bear + 2,000-path Monte Carlo.

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 20)
from data_pipeline import generate_synthetic_prices
from feature_engineering import FEATURE_COLUMNS, build_model_frame, time_split
from model import evaluate_binary_models, train_binary_models
from backtest import compare_strategies
frame = build_model_frame(generate_synthetic_prices())
train, test = time_split(frame)
models = train_binary_models(train[FEATURE_COLUMNS], train['target_up'])
results = evaluate_binary_models(models, test[FEATURE_COLUMNS], test['target_up'])
best_name = results.sort_values('f1', ascending=False).iloc[0]['model']
print('Best model by F1:', best_name)
metrics, curves = compare_strategies(test, models[best_name], FEATURE_COLUMNS)
metrics

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
for name, d in curves.items():
    ax.plot(d['date'], d['equity'], label=name)
ax.legend(); ax.set_title('Equity curves (growth of $1, synthetic)')
plt.show()

In [ ]:
from scenario import deterministic_scenarios, monte_carlo_summary
ml_daily = curves['ml_signal']
print(deterministic_scenarios(ml_daily['pnl']))
print(monte_carlo_summary(ml_daily['pnl']))

In [ ]:
import openpyxl
wb = openpyxl.load_workbook('../reports/kpi_dashboard.xlsx')
print('Dashboard sheets:', wb.sheetnames)